In [ ]:
# =================================================================
# CELL 1 — setup and CSV picker
# =================================================================
%matplotlib widget
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import glob
import ipywidgets as widgets
from IPython.display import display
 
 
def rr_density_phi(phi, sigma, k):
    l = 2.0 ** (-phi)
    x = (l / sigma) ** k
    return k * x * np.exp(-x)
 
 
def bi_rr_density(phi, w, sigma1, k1, sigma2, k2, amp):
    return amp * (w * rr_density_phi(phi, sigma1, k1) +
                  (1 - w) * rr_density_phi(phi, sigma2, k2))
 
 
csv_files = sorted(glob.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError("No .csv files found in the current folder.")
 
csv_picker = widgets.Dropdown(
    options=csv_files,
    description='CSV file:',
    style={'description_width': 'initial'},
)
display(csv_picker)
 
 

In [ ]:
# =================================================================
# CELL 2 — load data using the file selected above
# =================================================================
csv_path = csv_picker.value
print(f"Loading: {csv_path}")
 
df = pd.read_csv(csv_path, skiprows=1)
df.columns = ['phi', 'diameter_mm', 'wt_pct']
df = df.dropna(subset=['wt_pct']).reset_index(drop=True)
 
phi_data = df['phi'].values.astype(float)
wt_pct = df['wt_pct'].values.astype(float)
wt_pct_norm = wt_pct / wt_pct.sum() * 100  # ensure it sums to 100%
 
print(f"Loaded {csv_path}: {len(phi_data)} points, "
      f"Phi range [{phi_data.min():.2f}, {phi_data.max():.2f}]")
 

In [ ]:

# =================================================================
# CELL 3 — interactive picker
# =================================================================
picked_points = []  # reset each time this cell runs

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(phi_data, wt_pct_norm, width=0.85, color='lightgray',
       edgecolor='k', zorder=2)
ax.set_xlabel(r'$\phi = -\log_2$(diameter in mm)')
ax.set_ylabel('wt.% per phi bin')
ax.set_title('Click COARSE mode peak, then FINE mode peak')
ax.grid(alpha=0.3)

labels = ['coarse', 'fine']
colors = ['steelblue', 'seagreen']


def onclick(event):
    if event.inaxes != ax or len(picked_points) >= 2:
        return
    phi_click, y_click = event.xdata, event.ydata
    picked_points.append((phi_click, y_click))

    i = len(picked_points) - 1
    ax.axvline(phi_click, color=colors[i], linestyle='--', lw=1.5)
    ax.annotate(f'{labels[i]}\n$\\phi$={phi_click:.2f}',
                xy=(phi_click, y_click), xytext=(5, 5),
                textcoords='offset points', color=colors[i], fontsize=9)
    fig.canvas.draw_idle()

    if len(picked_points) == 2:
        print(f"\nBoth points picked:")
        print(f"  coarse: phi={picked_points[0][0]:.3f}, wt%={picked_points[0][1]:.3f}")
        print(f"  fine:   phi={picked_points[1][0]:.3f}, wt%={picked_points[1][1]:.3f}")
        print("You can now run the next cell to fit.")


cid = fig.canvas.mpl_connect('button_press_event', onclick)
plt.tight_layout()
plt.show()

In [ ]:
# =================================================================
# CELL 4 — build p0 from clicks and fit
# =================================================================
if len(picked_points) < 2:
    raise RuntimeError(
        "Only got {} click(s) so far. Click twice on the Cell 2 plot "
        "(coarse mode, then fine mode) before running this cell."
        .format(len(picked_points))
    )

(phi_coarse_pick, y_coarse_pick), (phi_fine_pick, y_fine_pick) = picked_points

sigma1_guess = 2.0 ** (-phi_coarse_pick)
sigma2_guess = 2.0 ** (-phi_fine_pick)
amp_guess = max(y_coarse_pick, y_fine_pick) * 2.0
w_guess = y_coarse_pick / (y_coarse_pick + y_fine_pick)

p0 = [w_guess, sigma1_guess, 4.0, sigma2_guess, 4.0, amp_guess]
lower = [0.0, 1e-4, 0.3, 1e-4, 0.3, 1]
upper = [1.0, 50.0, 15.0, 50.0, 15.0, 500]
p0 = list(np.clip(p0, lower, upper))

print(f"Initial guess built from clicks: {[round(x, 4) for x in p0]}")

ss_tot = np.sum((wt_pct_norm - wt_pct_norm.mean()) ** 2)

popt_bi, pcov_bi = curve_fit(
    bi_rr_density, phi_data, wt_pct_norm,
    p0=p0, bounds=(lower, upper), maxfev=50000
)
w_fit, sigma1_fit, k1_fit, sigma2_fit, k2_fit, amp_fit = popt_bi
fit_bi = bi_rr_density(phi_data, *popt_bi)
r2_bi = 1 - np.sum((wt_pct_norm - fit_bi) ** 2) / ss_tot

perr = np.sqrt(np.diag(pcov_bi))
w_err, s1_err, k1_err, s2_err, k2_err, amp_err = perr

if sigma1_fit < sigma2_fit:
    w_fit = 1 - w_fit
    sigma1_fit, sigma2_fit = sigma2_fit, sigma1_fit
    k1_fit, k2_fit = k2_fit, k1_fit
    s1_err, s2_err = s2_err, s1_err
    k1_err, k2_err = k2_err, k1_err

phi1_mode = -np.log2(sigma1_fit)
phi2_mode = -np.log2(sigma2_fit)
phi1_err = s1_err / (sigma1_fit * np.log(2))
phi2_err = s2_err / (sigma2_fit * np.log(2))

print("\nBi-Rosin-Rammler fit (from clicked starting guess):")
print(f"  w (coarse mass fraction) = {w_fit:.4f} +/- {w_err:.4f}")
print(f"  sigma_coarse             = {sigma1_fit:.5f} +/- {s1_err:.5f} mm"
      f"   (phi = {phi1_mode:.3f} +/- {phi1_err:.3f})")
print(f"  k_coarse                 = {k1_fit:.4f} +/- {k1_err:.4f}")
print(f"  sigma_fine                = {sigma2_fit:.5f} +/- {s2_err:.5f} mm"
      f"   (phi = {phi2_mode:.3f} +/- {phi2_err:.3f})")
print(f"  k_fine                   = {k2_fit:.4f} +/- {k2_err:.4f}")
print(f"  amp                      = {amp_fit:.4f} +/- {amp_err:.4f}")
print(f"  R^2                      = {r2_bi:.4f}")



In [ ]:
# =================================================================
# CELL 5 — plot the fit result
# =================================================================
phi_smooth = np.linspace(phi_data.min() - 0.5, phi_data.max() + 0.5, 500)
fit_bi_smooth = bi_rr_density(phi_smooth, w_fit, sigma1_fit, k1_fit, sigma2_fit, k2_fit, amp_fit)
fit_coarse_smooth = amp_fit * w_fit * rr_density_phi(phi_smooth, sigma1_fit, k1_fit)
fit_fine_smooth = amp_fit * (1 - w_fit) * rr_density_phi(phi_smooth, sigma2_fit, k2_fit)
 
fig2, axes2 = plt.subplots(3, 1, figsize=(8, 14))
 
# --- Mass density view (wt.% per phi bin) ---
ax2 = axes2[0]
ax2.bar(phi_data, wt_pct_norm, width=0.85, color='lightgray',
        edgecolor='k', label='data', zorder=2)
ax2.plot(phi_smooth, fit_bi_smooth, '-', color='crimson', lw=2,
         label=f'Bi-Rosin-Rammler ($R^2$={r2_bi:.3f})', zorder=4)
ax2.plot(phi_smooth, fit_coarse_smooth, '--', color='steelblue', label='Coarse mode')
ax2.plot(phi_smooth, fit_fine_smooth, '--', color='seagreen', label='Fine mode')
ax2.set_xlabel(r'$\phi = -\log_2$(diameter in mm)')
ax2.set_ylabel('wt.% per phi bin')
ax2.set_title('mass-density fit')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)
 
# --- Cumulative M(>l)/M_T view vs diameter (log scale) ---
df_sorted = df.sort_values('diameter_mm', ascending=False).reset_index(drop=True)
tot = df_sorted['wt_pct'].sum()
df_sorted['M_gt_l'] = 1 - df_sorted['wt_pct'].cumsum() / tot + df_sorted['wt_pct'] / tot
 
l_smooth = 2.0 ** (-phi_smooth)
order = np.argsort(phi_smooth)
cum_from_fine_end = np.cumsum(fit_bi_smooth[order])
cum_from_fine_end /= cum_from_fine_end[-1]
M_gt_l_fit_sorted = 1 - cum_from_fine_end
 
ax2 = axes2[1]
ax2.semilogx(df_sorted['diameter_mm'], df_sorted['M_gt_l'], 'o', color='k',
             label='data (empirical)', zorder=5)
ax2.semilogx(l_smooth[order], M_gt_l_fit_sorted, '-', color='crimson', lw=2,
             label='Bi-Rosin-Rammler (from fitted density)')
ax2.set_xlabel('Particle diameter, $l$ (mm, log scale)')
ax2.set_ylabel(r'$M(>l)/M_T$')
ax2.set_title('cumulative view (vs diameter)')
ax2.invert_xaxis()
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3, which='both')
 
# --- Cumulative M(>l)/M_T view vs phi ---
ax2 = axes2[2]
ax2.plot(df_sorted['phi'], df_sorted['M_gt_l'], 'o', color='k',
         label='data (empirical)', zorder=5)
ax2.plot(phi_smooth[order], M_gt_l_fit_sorted, '-', color='crimson', lw=2,
         label='Bi-Rosin-Rammler (from fitted density)')
ax2.set_xlabel(r'$\phi = -\log_2$(diameter in mm)')
ax2.set_ylabel(r'$M(>l)/M_T$')
ax2.set_title('cumulative view (vs phi)')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3, which='both')
 
plt.tight_layout()
plt.show()


print("\nBi-Rosin-Rammler fit (from clicked starting guess):")
print(f"  w (coarse mass fraction) = {w_fit:.4f} +/- {w_err:.4f}")
print(f"  sigma_coarse             = {sigma1_fit:.5f} +/- {s1_err:.5f} mm"
      f"   (phi = {phi1_mode:.3f} +/- {phi1_err:.3f})")
print(f"  k_coarse                 = {k1_fit:.4f} +/- {k1_err:.4f}")
print(f"  sigma_fine                = {sigma2_fit:.5f} +/- {s2_err:.5f} mm"
      f"   (phi = {phi2_mode:.3f} +/- {phi2_err:.3f})")
print(f"  k_fine                   = {k2_fit:.4f} +/- {k2_err:.4f}")
print(f"  amp                      = {amp_fit:.4f} +/- {amp_err:.4f}")
print(f"  R^2                      = {r2_bi:.4f}")